In [1]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SpatialDropout1D, LSTM, Dense
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from nltk.corpus import stopwords
import nltk


In [2]:
#downloading the stopwords so we would not have to define them manually
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [33]:
import pandas as pd

# Loading the data from csv files.
train_path = "/content/drive/MyDrive/For Collab/Quiz_and_assignments/imdb_twitter/twitter_training.csv"
val_path = "/content/drive/MyDrive/For Collab/Quiz_and_assignments/imdb_twitter/twitter_validation.csv"

df_train = pd.read_csv(train_path, encoding='utf-8') #trying with the encoding
df_val = pd.read_csv(val_path, encoding='utf-8')

print("Train Data Shape:", df_train.shape)
print("Validation Data Shape:", df_val.shape)


Train Data Shape: (74681, 4)
Validation Data Shape: (999, 4)


In [34]:
# CHecking the Column names
print(df_train.columns)
print(df_val.columns)


Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')
Index(['3364', 'Facebook', 'Irrelevant',
       'I mentioned on Facebook that I was struggling for motivation to go for a run the other day, which has been translated by Tom’s great auntie as ‘Hayley can’t get out of bed’ and told to his grandma, who now thinks I’m a lazy, terrible person 🤣'],
      dtype='object')


In [35]:
import pandas as pd

#adding correct column names to the data
column_names = ["id", "candidate", "sentiment", "text"]

df_train = pd.read_csv("/content/drive/MyDrive/For Collab/Quiz_and_assignments/imdb_twitter/twitter_training.csv", names=column_names, header=None)
df_val = pd.read_csv("/content/drive/MyDrive/For Collab/Quiz_and_assignments/imdb_twitter/twitter_validation.csv", names=column_names, header=None)

# Print to check if the columns are correct now
print(df_train.head())
print(df_train.head())


     id    candidate sentiment  \
0  2401  Borderlands  Positive   
1  2401  Borderlands  Positive   
2  2401  Borderlands  Positive   
3  2401  Borderlands  Positive   
4  2401  Borderlands  Positive   

                                                text  
0  im getting on borderlands and i will murder yo...  
1  I am coming to the borders and I will kill you...  
2  im getting on borderlands and i will kill you ...  
3  im coming on borderlands and i will murder you...  
4  im getting on borderlands 2 and i will murder ...  
     id    candidate sentiment  \
0  2401  Borderlands  Positive   
1  2401  Borderlands  Positive   
2  2401  Borderlands  Positive   
3  2401  Borderlands  Positive   
4  2401  Borderlands  Positive   

                                                text  
0  im getting on borderlands and i will murder yo...  
1  I am coming to the borders and I will kill you...  
2  im getting on borderlands and i will kill you ...  
3  im coming on borderlands and i will m

In [36]:
# checking the column names.
print(df_train.columns)
print(df_val.columns)


Index(['id', 'candidate', 'sentiment', 'text'], dtype='object')
Index(['id', 'candidate', 'sentiment', 'text'], dtype='object')


In [37]:
# Joining both the train and validation dataset for preprocessing.
df = pd.concat([df_train, df_val], ignore_index=True)


In [38]:
df.columns

Index(['id', 'candidate', 'sentiment', 'text'], dtype='object')

In [39]:
df.head()

,id,candidate,sentiment,text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75682 entries, 0 to 75681
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         75682 non-null  int64 
 1   candidate  75682 non-null  object
 2   sentiment  75682 non-null  object
 3   text       74996 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [41]:
# Keeping relevant columns
df = df[['candidate', 'sentiment', 'text']]


In [42]:
# Preprocessing function (to lower case, removing the punctuation and removal of stop wrods)
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords

# Download stopwords if not already done
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Preprocessing function
def preprocess_text(text):
    if not isinstance(text, str):  # Check if text is not a string
        return ""  # Return an empty string for NaN or float values

    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove punctuation
    text = ' '.join(word for word in text.split() if word not in stop_words)  # Remove stopwords
    return text

# Apply the function safely
df['text'] = df['text'].astype(str)  # Convert all values in 'text' column to string
df['clean_text'] = df['text'].apply(preprocess_text)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [43]:
df['clean_text'] = df['text'].apply(preprocess_text)



In [44]:
# Convert sentiment labels to numerical format


from tensorflow.keras.utils import to_categorical

# Define a mapping for sentiment labels
sentiment_mapping = {
    "Positive": 0,
    "Negative": 1,
    "Irrelevant": 2
}

# Convert sentiment column to numeric values
df['sentiment'] = df['sentiment'].map(sentiment_mapping)

# Handle any NaN values (if there were labels not in the mapping)
df = df.dropna(subset=['sentiment'])  # Drop rows where sentiment couldn't be mapped

# Convert to integer type
df['sentiment'] = df['sentiment'].astype(int)

# Now convert to categorical
y = to_categorical(df['sentiment'], num_classes=3)


<ipython-input-44-4fc3e3e88ea5>:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment'] = df['sentiment'].astype(int)


In [45]:
# Tokenization and padding
max_words = 5000
max_len = 100
tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(df['clean_text'])
X = tokenizer.texts_to_sequences(df['clean_text'])
X = pad_sequences(X, maxlen=max_len, padding='post')

In [46]:
# Converting labels to categorical format
y = to_categorical(df['sentiment'], num_classes=3)

In [47]:
# Spliting dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [54]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SpatialDropout1D, LSTM, Bidirectional, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

# Define optimized model
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),  # Increased embedding size
    SpatialDropout1D(0.3),  # Prevent overfitting
    Bidirectional(LSTM(64, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)),  # Stacked BiLSTM
    BatchNormalization(),
    Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3)),  # Second BiLSTM layer
    Dense(64, activation='relu'),
    Dropout(0.4),  # Regularization
    Dense(3, activation='softmax')  # Multi-class classification
])

# Use Adam optimizer with learning rate decay
optimizer = Adam(learning_rate=0.001, decay=1e-6)

model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Train with GPU acceleration and batch size tuning
with tf.device('/GPU:0'):  # Ensure using GPU
    model.fit(X_train, y_train,
              batch_size=512,  # Larger batch size for speed
              epochs=10,
              validation_data=(X_test, y_test),
              verbose=1)


Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/optimizers/base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


90/90 ━━━━━━━━━━━━━━━━━━━━ 384s 4s/step - accuracy: 0.4125 - loss: 1.0727 - val_accuracy: 0.6336 - val_loss: 1.0090
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 378s 4s/step - accuracy: 0.7115 - loss: 0.6947 - val_accuracy: 0.6917 - val_loss: 0.8975
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 361s 4s/step - accuracy: 0.7903 - loss: 0.5262 - val_accuracy: 0.7553 - val_loss: 0.7841
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 322s 4s/step - accuracy: 0.8243 - loss: 0.4418 - val_accuracy: 0.7858 - val_loss: 0.6464
Epoch 5/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 332s 4s/step - accuracy: 0.8395 - loss: 0.4012 - val_accuracy: 0.8014 - val_loss: 0.5221
Epoch 6/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 379s 4s/step - accuracy: 0.8554 - loss: 0.3590 - val_accuracy: 0.8233 - val_loss: 0.4433
Epoch 7/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 370s 4s/step - accuracy: 0.8677 - loss: 0.3296 - val_accuracy: 0.8353 - val_loss: 0.4107
Epoch 8/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 326s 4s/step - accuracy: 0.8707 - loss: 0.3122 - val_accuracy: 0.8441 - val_loss: 0.

In [ ]:
# Evaluating the model accuracy on validation data.
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {accuracy:.4f}')


In [50]:
#print(df['sentiment'].value_counts(normalize=True))


sentiment
1    0.399587
0    0.369821
2    0.230593
Name: proportion, dtype: float64
